In [4]:
import torch
import torch.nn as nn
torch.manual_seed(42)
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

In [5]:
def make_lag_df(df, feature_cols, window=14):
    out = {}
    for col in feature_cols:
        s = df[col]
        for lag in range(window, 0, -1):
            out[f"{col}_t-{lag}"] = s.shift(lag)
    df_seq = pd.DataFrame(out, index=df.index).dropna()
    return df_seq

def df_to_loader(df_seq, feature_cols, lagged_feature_cols, target_col="TargetDelta",
                 window=14, batch_size=128, shuffle=False):

    # Create input feature sequences X with shape (N, T, F)
    Xflat = torch.tensor(df_seq[lagged_feature_cols].to_numpy(), dtype=torch.float32) 
    N = Xflat.shape[0]
    T = window
    F = len(feature_cols)
    X = Xflat.view(N, T, F)

    # Create targets y with shape (N, 1)
    y = torch.tensor(df_seq[target_col].to_numpy(), dtype=torch.float32).unsqueeze(1) 

    # Create TensorDataset and DataLoader
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return loader

# Load testing set
spy_test_df = pd.read_csv("./data/spy_test.csv", index_col="Date")

features = ['Open', 'High', 'Low', 'Close', 'Volume', 'sma_10', 'ema_20', 'rsi_14',
       'macd', 'macd_signal', 'macd_hist', 'atr_14', 'bb_upper', 'bb_lower']

spy_test_seq_df = make_lag_df(spy_test_df, features, window=24)
lagged_features = spy_test_seq_df.columns

import joblib
scaler = joblib.load("./models/scaler_spy.pkl")
spy_test_seq_df_scaled = pd.DataFrame(scaler.transform(spy_test_seq_df), 
                                      index=spy_test_seq_df.index, 
                                      columns=spy_test_seq_df.columns)

spy_test_seq_df_scaled["TargetDelta"] = spy_test_df.loc[spy_test_seq_df_scaled.index, "TargetDelta"].astype("float32")

test_loader = df_to_loader(spy_test_seq_df_scaled, 
                            feature_cols=features,
                            lagged_feature_cols=lagged_features,
                            target_col="TargetDelta",
                            window=24,
                            batch_size=128, 
                            shuffle=False,)

spy_test_df.round(2).head()

c:\Users\nomadant\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Open,High,Low,Close,Volume,sma_10,ema_20,rsi_14,macd,macd_signal,macd_hist,atr_14,bb_upper,bb_lower,Target,TargetDelta
Date,,,,,,,,,,,,,,,,
2022-06-13,361.49,363.35,355.26,356.87,170004900,385.94,384.09,37.91,-4.76,-3.27,-1.49,8.83,404.53,360.95,355.80,-1.08
2022-06-14,358.63,359.67,352.68,355.80,104011800,382.22,381.40,38.72,-6.45,-3.90,-2.54,8.65,406.39,356.58,360.87,5.07
2022-06-15,359.12,365.34,354.13,360.87,125666800,379.33,379.44,39.97,-7.29,-4.58,-2.71,8.94,406.39,353.82,348.93,-11.94
2022-06-16,352.60,353.01,346.48,348.93,134473300,374.50,376.54,29.70,-8.82,-5.43,-3.39,9.31,408.55,349.26,349.68,0.75
2022-06-17,349.34,353.04,346.15,349.68,111113900,370.40,373.98,22.22,-9.85,-6.31,-3.54,9.12,410.10,345.61,358.48,8.80


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SimpleRNN(nn.Module):
    def __init__(self, input_size=14, hidden=128, num_layers=2):
        super().__init__()
        self.rnn = nn.RNN(input_size=input_size, 
                          hidden_size=hidden, 
                          num_layers=num_layers, 
                          batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        out, hidden = self.rnn(x)           
        return self.fc(out[:, -1, :])   
        
## YOUR SOLUTION HERE ##
class SimpleGRU(nn.Module):
    def __init__(self, input_size=14, hidden=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, 
                          hidden_size=hidden, 
                          num_layers=num_layers, 
                          batch_first=True)
        
        self.fc = nn.Linear(hidden, 1)
        
    def forward(self, x):
        out, hidden = self.gru(x)          
        return self.fc(out[:, -1, :]) 

class SimpleLSTM(nn.Module):
    def __init__(self, input_size=14, hidden=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, 
                            hidden_size=hidden, 
                            num_layers=num_layers, 
                            batch_first=True)
        
        self.fc = nn.Linear(hidden, 1)
        
    def forward(self, x):
        out, _ = self.lstm(x)         
        return self.fc(out[:, -1, :]) 

# Instantiate models + pre-trained weights
RNN_PATH = "./models/rnn_spy_state.pt"
GRU_PATH  = "./models/gru_spy_state.pt"
LSTM_PATH = "./models/lstm_spy_state.pt"

rnn = SimpleRNN(input_size=14, hidden=128, num_layers=2).to(device)
rnn_state = torch.load(RNN_PATH, map_location=device)
rnn.load_state_dict(rnn_state)

gru  = SimpleGRU(input_size=14, hidden=128, num_layers=2).to(device)
gru_state = torch.load(GRU_PATH, map_location=device)
gru.load_state_dict(gru_state)

lstm = SimpleLSTM(input_size=14, hidden=128, num_layers=2).to(device)
lstm_state = torch.load(LSTM_PATH, map_location=device)
lstm.load_state_dict(lstm_state)

<All keys matched successfully>

In [7]:
def predict(model, dataloader):
    model.eval()
    batch_preds = []
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            batch_pred = model(X_batch.to(device))
            batch_pred = batch_pred.cpu().detach().numpy().ravel()
            batch_preds.append(batch_pred)
    y_preds = np.concatenate(batch_preds)
    return y_preds

rnn_preds = predict(rnn, test_loader)
gru_preds = predict(gru, test_loader)
lstm_preds = predict(lstm, test_loader)

# Show output - Test set MSE performance
from sklearn.metrics import mean_squared_error
y_true = spy_test_seq_df_scaled["TargetDelta"].to_numpy()
mse_rnn  = mean_squared_error(y_true, rnn_preds)
mse_gru  = mean_squared_error(y_true, gru_preds)
mse_lstm = mean_squared_error(y_true, lstm_preds)

print("=== Test Performance ===")
print(f"RNN  MSE: {mse_rnn:.6f}")
print(f"GRU  MSE: {mse_gru:.6f}")
print(f"LSTM MSE: {mse_lstm:.6f}")

=== Test Performance ===
RNN  MSE: 32.896969
GRU  MSE: 43.253563
LSTM MSE: 32.933613


In [8]:
# Obtain the most recent prediction
pred_delta_rnn  = rnn_preds[-1]
pred_delta_gru  = gru_preds[-1]
pred_delta_lstm = lstm_preds[-1]

# Use delta + prev close price to calculate the new predicted close price
prev_close = float(spy_test_df["Close"].iloc[-1])  
actual_close = float(spy_test_df["Target"].iloc[-1]) 

## YOUR SOLUTION HERE ##
pred_next_close_rnn  = prev_close + pred_delta_rnn
pred_next_close_gru  = prev_close + pred_delta_gru
pred_next_close_lstm = prev_close + pred_delta_lstm

# Show output - Predicted next-day close
print(f"Previous Close:       ${prev_close:.2f}")
print(f"RNN:  Pred Δ Close:    {pred_delta_rnn:+.2f}  | Next Close: ${pred_next_close_rnn:.2f}")
print(f"GRU:  Pred Δ Close:    {pred_delta_gru:+.2f}  | Next Close: ${pred_next_close_gru:.2f}")
print(f"LSTM: Pred Δ Close:    {pred_delta_lstm:+.2f} | Next Close: ${pred_next_close_lstm:.2f}")
print(f"Actual Close:         ${actual_close:.2f}")

Previous Close:       $663.70
RNN:  Pred Δ Close:    -4.07  | Next Close: $659.63
GRU:  Pred Δ Close:    -3.44  | Next Close: $660.26
LSTM: Pred Δ Close:    +5.38 | Next Close: $669.08
Actual Close:         $666.61
